# world-cup-climate — simple (IFS only)

For today's World Cup match, compare **temperature** and **heat index** (how hot it
feels) at the **match venue** vs **each country's capital**.

One data source: `spring-data/ecwmf-ifs-15-days-forecast-open` (ECMWF IFS, open).
- **best estimate** = `step="0 days"` across every init (recent conditions up to today)
- **forecast** = the latest init, `0 … 15 days` ahead

No ERA5, no stitching, no climatology — just IFS.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from world_cup_climate.fixtures import load_matches
from world_cup_climate.ifs import latest_init, location_series, matchday_value
from world_cup_climate import viz_ifs

print("latest IFS init:", latest_init())
matches = load_matches("2026-06-15")
for i, m in enumerate(matches):
    print(f"[{i}] {m.title:26s} @ {m.venue.name}")
match = matches[0]   # Brazil vs Croatia @ AT&T Stadium (Dallas)
match.title, match.venue.name

## Temperature — venue vs both capitals

In [ ]:
viz_ifs.plot_match(match.places(), col='t2m_c', matchday=match.date); plt.show()

## Heat index (feels-like) — the sports-relevant heat-stress number

In [ ]:
viz_ifs.plot_match(match.places(), col='heat_index_c', matchday=match.date); plt.show()

## Matchday snapshot — daily max at each location

In [ ]:
import pandas as pd
rows = []
for p in match.places():
    s = location_series(p.lat, p.lon)
    rows.append({
        "role": p.label.split(" — ")[0],
        "location": p.name,
        "t2m_max [C]": round(matchday_value(s, match.date, "t2m_c"), 1),
        "heat_index_max [C]": round(matchday_value(s, match.date, "heat_index_c"), 1),
    })
pd.DataFrame(rows).set_index("location")

## Ideas / next steps
See `README.md`: heat index via [`xclim`](https://xclim.readthedocs.io/en/stable/api_indicators.html#xclim.indicators.convert.heat_index),
and ERA5 (`earthmover-public/era5-private`) for a 10-year historical comparison.